# Webhooks (signed, retried, idempotent) + OpenAPI contract tests

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 26 §26.5–§26.6 · type: walkthrough*

**One-line promise:** deliver outbound events the way you'd want to receive them — HMAC-signed, retried with backoff, idempotent on the consumer — then turn FastAPI's generated `openapi.json` into a contract test that fails the moment a field is renamed. All in-process.

## 🧠 Why this matters

Not every interaction is request→response. Sometimes *your* system must notify *theirs* — a long agent run finished, a document finished processing. That's a **webhook**: you `POST` an event to a URL the consumer registered. Three properties make webhooks trustworthy, and they're the same rigor you apply to your inbound API: **sign** them (so the receiver knows it's really you), **retry** them (the receiver may be briefly down — at-least-once delivery), and make the consumer **idempotent** (because at-least-once means you *will* send duplicates).

The other source of truth in an enterprise API is the **OpenAPI spec** FastAPI generates for free. Pin it as a contract and a CI test catches breaking changes *before* they ship and break every client.

## Objectives & prereqs

**By the end you can:**
- Sign a webhook payload with an **HMAC** and have a receiver reject a *tampered* body.
- Deliver with **retry + backoff** to a flaky receiver and reason about at-least-once.
- Make the *consumer* **idempotent** with an idempotency key so a redelivered event is harmless (the Ch 24 pattern).
- Dump FastAPI's `openapi.json` and write a **contract test** that fails when a field is renamed or removed.

**Prereqs:** notebook **26-02**; Ch 24 (idempotency). Foreshadows Ch 29 (retries / at-least-once) and Ch 31 (n8n automations consuming these events). No network, no real outbound HTTP — receiver and delivery loop are in-process; the clock is faked.

In [ ]:
# --- Setup: imports, env, and the MOCK switch ---------------------------------
# stdlib only (+ python-dotenv & fastapi from requirements.txt). No outbound HTTP.
import os
import json
import hmac
import hashlib
import random
import pathlib

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

MOCK = os.getenv("COMPANION_MOCK", "1") == "1"
random.seed(26)  # deterministic 'flaky receiver' schedule below

# The signing secret comes from the environment ONLY. Dev fallback so the notebook runs
# with no setup; a real sender shares this secret with the receiver out of band.
WEBHOOK_SECRET = os.getenv("WEBHOOK_SECRET", "whsec_dev-only-not-real")

DATA = pathlib.Path("data")
sample_event = json.loads((DATA / "webhook-event.json").read_text())
print(f"MOCK mode: {MOCK}  | webhook secret from env: {'WEBHOOK_SECRET' in os.environ}")
print("sample event:", sample_event["type"], "->", sample_event["data"]["run_id"])

## 1 · Sign the payload so the receiver can trust it (§26.6, 🔧 Build)

A webhook arrives at a public URL — anyone could `POST` to it. The fix is a shared-secret **HMAC** over the *exact bytes* of the body, sent in a header (Stripe/GitHub call it `X-Signature`). The receiver recomputes the HMAC over what it received and compares in constant time. Any change to the body changes the signature, so a tampered payload is rejected.

In [ ]:
def sign_payload(body: bytes, secret: str) -> str:
    """HMAC-SHA256 over the raw body bytes -> hex digest (the X-Signature header)."""
    return hmac.new(secret.encode(), body, hashlib.sha256).hexdigest()

def verify_signature(body: bytes, signature: str, secret: str) -> bool:
    expected = sign_payload(body, secret)
    return hmac.compare_digest(expected, signature)  # constant-time

# Serialize ONCE; sign and send the same bytes (re-serializing can reorder keys!).
body = json.dumps(sample_event, separators=(",", ":"), sort_keys=True).encode()
sig = sign_payload(body, WEBHOOK_SECRET)
print("signature:", sig[:32], "...")
print("verify untampered:", verify_signature(body, sig, WEBHOOK_SECRET))

🔮 **Predict.** An attacker intercepts the event and flips the run's `status` from `succeeded` to `failed`, but **can't** recompute the HMAC (no secret). Does `verify_signature` on the tampered body return `True` or `False`? Decide before running.

In [ ]:
tampered = dict(sample_event)
tampered_data = dict(sample_event["data"])
tampered_data["status"] = "failed"            # the attacker's edit
tampered["data"] = tampered_data
tampered_body = json.dumps(tampered, separators=(",", ":"), sort_keys=True).encode()

# The attacker forwards the ORIGINAL signature (they can't make a valid new one).
print("verify tampered body w/ original sig:", verify_signature(tampered_body, sig, WEBHOOK_SECRET))

**What you just saw.** `False` — the receiver recomputes the HMAC over the *bytes it got*, which no longer match the signature, so it rejects the event. Without the secret, an attacker can read or replay but cannot *forge*. (Replay is handled next, by idempotency.)

## 2 · Deliver with retry + backoff to a flaky receiver (§26.6, 🔧 Build)

The receiver might be briefly down (deploy, blip). **At-least-once** delivery means: retry with backoff until it succeeds or you exhaust the budget — then dead-letter it. We model a receiver that fails its first two attempts (e.g. `503`) and succeeds on the third, and a sender that backs off between tries (clock faked, no real sleeping).

In [ ]:
class FlakyReceiver:
    """Verifies the signature, then fails `fail_times` before succeeding (503)."""
    def __init__(self, fail_times: int, secret: str):
        self.left = fail_times
        self.secret = secret
        self.deliveries = []   # bodies that were ACCEPTED (after a 200)
    def receive(self, body: bytes, signature: str) -> int:
        if not verify_signature(body, signature, self.secret):
            return 401  # bad signature -> reject, do NOT process
        if self.left > 0:
            self.left -= 1
            return 503  # transient: please retry
        self.deliveries.append(body)
        return 200

def deliver_with_retry(receiver, body, signature, *, max_attempts=5, base=1.0):
    """At-least-once delivery: retry transient failures with exponential backoff."""
    delays = []
    for attempt in range(1, max_attempts + 1):
        status = receiver.receive(body, signature)
        if status == 200:
            return {"ok": True, "attempts": attempt, "delays": delays}
        if status == 401:
            return {"ok": False, "attempts": attempt, "reason": "bad signature (no retry)"}
        if attempt < max_attempts:
            backoff = base * (2 ** (attempt - 1))
            delays.append(backoff)        # real sender would time.sleep(backoff)
    return {"ok": False, "attempts": max_attempts, "reason": "exhausted (dead-letter it)"}

receiver = FlakyReceiver(fail_times=2, secret=WEBHOOK_SECRET)
print("delivering... (receiver fails twice, then 200)")

🔮 **Predict.** The receiver fails the first **2** attempts, then returns `200`. How many *total* attempts before success, and what backoff delays were used between them (base 1s, doubling)? Decide before running.

In [ ]:
result = deliver_with_retry(receiver, body, sig)
print(result)
print("accepted deliveries at receiver:", len(receiver.deliveries))

**What you just saw.** Three attempts: fail (wait 1s), fail (wait 2s), succeed. The event was delivered exactly once *to the application* here — but at-least-once means that in the real world a `200` can be lost on the way back, and you'd retry an *already-processed* event. That's the next problem.

## 3 · ⚠️ Pitfall: a non-idempotent consumer (the Ch 24 fix, 🔧 Build)

At-least-once delivery **guarantees** the consumer will eventually see duplicates — a retried event whose previous `200` was lost in transit. If the handler isn't idempotent, a duplicate "run completed" event might charge the customer twice or kick off the downstream automation twice. The fix is the Ch 24 idempotency key: dedupe on the event id so a redelivery is a harmless no-op.

In [ ]:
# A non-idempotent handler: every delivery triggers the side effect.
charges_naive = []
def handle_naive(event):
    charges_naive.append(event["data"]["run_id"])   # side effect: bill the run

# Same event delivered THREE times (at-least-once duplicates).
for _ in range(3):
    handle_naive(sample_event)
print("naive handler -> charges fired:", len(charges_naive), "(duplicated!)")

# The idempotent handler: dedupe on the event id (its idempotency key).
_seen = set()
charges_idem = []
def handle_idempotent(event):
    key = event["id"]                 # stable per logical event, not per delivery
    if key in _seen:
        return  # already processed -> no-op (the whole point)
    _seen.add(key)
    charges_idem.append(event["data"]["run_id"])

for _ in range(3):
    handle_idempotent(sample_event)
print("idempotent handler -> charges fired:", len(charges_idem), "(exactly once)")
assert len(charges_idem) == 1, "idempotency must collapse redeliveries to one effect"

**What you just saw.** Same three deliveries: the naive handler billed the run **3×**; the idempotent handler billed it **once** and ignored the replays. Sign + retry + idempotent is the trio — you can't safely have at-least-once delivery without an idempotent consumer.

## 4 · The OpenAPI spec as a contract test (§26.5, 🔧 Build)

FastAPI generates an **OpenAPI** description of every endpoint for free. Treat it as the single source of truth: snapshot it, and a tiny **contract test** fails the build the moment a field or path is renamed/removed — catching a breaking change in CI before it ships and breaks every client (and it's the basis for generated SDKs). We build a small app, dump its `openapi.json`, and diff it against the committed baseline in `data/`.

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="Agent API", version="1.0.0")

class RunRequest(BaseModel):
    agent: str
    prompt: str

class Run(BaseModel):
    run_id: str
    status: str

@app.post("/v1/runs", operation_id="create_run")
def create_run(req: RunRequest) -> Run:
    return Run(run_id="run_8a21", status="queued")

@app.get("/v1/runs/{run_id}", operation_id="get_run")
def get_run(run_id: str) -> Run:
    return Run(run_id=run_id, status="succeeded")

spec = app.openapi()
print("generated paths:", sorted(spec["paths"].keys()))
print("operationIds:", sorted(op["operationId"]
                              for p in spec["paths"].values() for op in p.values()))

In [ ]:
# A focused contract test: paths + operationIds + path params must not silently change.
def contract_surface(spec: dict) -> set:
    """Reduce a spec to the surface clients depend on: (method, path, operationId)."""
    surface = set()
    for path, ops in spec["paths"].items():
        for method, op in ops.items():
            surface.add((method.upper(), path, op.get("operationId")))
    return surface

baseline = json.loads((DATA / "openapi-baseline.json").read_text())
current = contract_surface(spec)
expected = contract_surface(baseline)

missing = expected - current      # something the baseline promised is gone -> BREAKING
added = current - expected        # new surface -> usually safe, but flag it
print("removed/renamed (BREAKING):", missing or "none")
print("added (review):", added or "none")
assert not missing, f"contract broke: {missing}"
print("\ncontract test PASSED -- current API still satisfies the committed contract")

🔮 **Predict.** Suppose a teammate renames the field `run_id` to `id` (or changes `operation_id="get_run"` to `"fetch_run"`). Will the contract test above pass or fail, and *which* set — `missing` or `added` — catches it? Think it through, then try it in Exercise 2.

**What you just saw.** The current API still satisfies the committed contract, so the test passes. Rename a path or `operationId` and `missing` becomes non-empty — the assertion fails, the CI job goes red, and the breaking change is stopped *before* it reaches a single consumer. This same spec is what SDK generators consume to emit typed clients.

## 🎯 Senior lens

Treat **outbound** events with the same rigor as your **inbound** API. The asymmetry is seductive: the inbound API is obviously a contract — it has docs, tests, versioning — while webhooks feel like fire-and-forget. They aren't. An unsigned webhook is an open door; an un-retried one silently drops events the first time the receiver hiccups; a non-idempotent consumer double-charges on the inevitable duplicate. Each of those quietly erodes the trust that makes an integration valuable.

So: sign them, retry with backoff and a dead-letter, make consumers idempotent, version the payload (`type` + a schema), and — yes — contract-test the *event* shape too, not just the REST surface. The difference between a platform partners build on and one they rip out is exactly this rigor applied to the events nobody sees until they break.

## Recap

- A webhook is *your* system `POST`-ing an event to a URL the consumer registered — for work that finishes asynchronously (a long agent run).
- **Sign** the raw body with an HMAC; the receiver recomputes and compares in constant time, rejecting any tampered payload.
- **Retry with backoff** for at-least-once delivery; exhaustion means dead-letter, not silent drop.
- At-least-once **guarantees** duplicates, so the **consumer must be idempotent** — dedupe on the event id (the Ch 24 key).
- FastAPI's generated **OpenAPI** spec is a contract: snapshot it and a tiny test fails CI when a path/field is renamed or removed — and it powers SDK generation.
- Outbound events deserve the same rigor as the inbound API: signed, retried, idempotent, versioned, contract-tested.

## Exercises

Predict the result before running each.

1. **Exhaust the budget.** Set `FlakyReceiver(fail_times=10)` with `max_attempts=5`. What does `deliver_with_retry` return, and what are the backoff delays? Where would a real system send the event next (hint: dead-letter queue)?
2. **Break the contract.** Rename `operation_id="get_run"` to `"fetch_run"`, regenerate the spec, and re-run the contract test. Which set (`missing`/`added`) trips, and what exception surfaces?
3. **Replay window.** Add a `timestamp` to the signature input and have the receiver reject events older than 5 minutes. Why does signing the timestamp (not just the body) matter for replay protection?
4. **Idempotency key choice.** Change `handle_idempotent` to key on `event["data"]["run_id"]` instead of `event["id"]`. Construct two *different* events for the same run and predict whether the second is wrongly dropped. Why must the key match the *logical event*, not the entity?

In [ ]:
# Exercise 1 -- your code here


In [ ]:
# Exercise 2 -- your code here


In [ ]:
# Exercise 3 -- your code here


In [ ]:
# Exercise 4 -- your code here


## Next

- ⬅️ **Previous:** [`26-02-rate-limiting-quotas-errors-pagination.ipynb`](./26-02-rate-limiting-quotas-errors-pagination.ipynb).
- ➡️ **Part VIII** takes this enterprise-grade API to the cloud; **Ch 29** revisits retries / at-least-once in depth, and **Ch 31** consumes these webhooks in n8n automations.
- 📓 **Book:** §26.5 (OpenAPI, contract testing, SDKs) and §26.6 (webhooks, event-driven APIs).
- 🧱 **Template:** the webhook sender (sign + retry) and contract test become defaults in [`templates/fastapi-agent-service/`](../../templates/fastapi-agent-service/).
- 🏗️ **Capstone:** signed webhooks connect `capstone-project/app/` to `capstone-project/workers/` automations (checkpoint `ch26-enterprise-api`).